# Task 2.10 — Chọn mô hình hoặc ensemble DL bằng OOF

**Inputs:** competition data + outputs **2.6 và 2.9**. Không huấn luyện; có thể tắt GPU.
Chỉ thử mô hình đơn và ba blend trọng số định trước (25/75, 50/50, 75/25) giữa hai ứng viên có mean fold AUC cao nhất.
Mặc định chỉ thay baseline nếu tăng mean AUC ít nhất 0.0001 và thắng ít nhất 3/5 folds. Đây là ngưỡng thực hành để tránh chọn chênh lệch rất nhỏ, **không phải kiểm định ý nghĩa thống kê**.
Không đảm bảo ensemble sẽ tốt hơn; nếu không đủ bằng chứng, giữ mô hình đơn hoặc baseline.


In [ ]:
from pathlib import Path
import importlib.util
import sys
import hashlib
import json

DATA_DIR = None  # Auto-detect inputs; set a directory here only to disambiguate.

def resolve_data_dir(data_dir=None, expected_sha256=None,
                     required_files=("train.csv",), input_root="/kaggle/input"):
    """Find attached data without assuming Kaggle's mount layout."""
    from pathlib import Path
    import hashlib

    root = Path(input_root)
    if data_dir is not None:
        candidates = [Path(data_dir)]
        missing = [name for name in required_files if not (candidates[0] / name).is_file()]
        if missing:
            raise FileNotFoundError(
                f"DATA_DIR={data_dir} is missing {missing}. "
                "Attach competition data or set DATA_DIR=None for automatic discovery."
            )
    else:
        candidates = sorted({
            path.parent for path in root.rglob("train.csv")
            if path.is_file() and all((path.parent / name).is_file() for name in required_files)
        })
        if not candidates:
            mounted = sorted(path.name for path in root.iterdir()) if root.is_dir() else []
            raise FileNotFoundError(
                f"No attached directory contains {list(required_files)} under {root}. "
                "Use Add Input to attach competition playground-series-s6e11. "
                f"Mounted entries: {mounted}"
            )

    if expected_sha256 is not None:
        matching = []
        for directory in candidates:
            digest = hashlib.sha256()
            with (directory / "train.csv").open("rb") as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    digest.update(block)
            if digest.hexdigest() == expected_sha256:
                matching.append(directory)
        if not matching:
            raise ValueError(
                "Attached train.csv does not match the task 2.6 checksum. "
                f"Check the competition data/version. Candidates: {candidates}"
            )
        candidates = matching
    elif len(candidates) > 1:
        bundles = [directory for directory in candidates
                   if all((directory / name).is_file()
                          for name in ("train.csv", "test.csv", "sample_submission.csv"))]
        if bundles:
            candidates = bundles

    if len(candidates) != 1:
        raise ValueError(
            f"Multiple valid data directories found: {candidates}. "
            "Set DATA_DIR to the exact directory you want to use."
        )
    return candidates[0]

# Nếu gắn nhiều version 2.6, điền chính xác thư mục version cần dùng.
FOUNDATION_DIR = None
if FOUNDATION_DIR is None:
    found = sorted(Path("/kaggle/input").rglob("foundation_manifest.json"))
    if len(found) != 1:
        raise FileNotFoundError(f"Attach exactly one task 2.6 output; found: {found}")
    FOUNDATION_DIR = found[0].parent
else:
    FOUNDATION_DIR = Path(FOUNDATION_DIR)
manifest = json.loads((FOUNDATION_DIR / "foundation_manifest.json").read_text(encoding="utf-8"))
engine_file = FOUNDATION_DIR / "credit_fraud_dl_engine.py"
if hashlib.sha256(engine_file.read_bytes()).hexdigest() != manifest["engine_sha256"]:
    raise ValueError("Foundation engine checksum mismatch.")
spec = importlib.util.spec_from_file_location("credit_fraud_dl_engine", engine_file)
dl = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = dl
spec.loader.exec_module(dl)

import numpy as np
import pandas as pd
DATA_DIR = resolve_data_dir(DATA_DIR, expected_sha256=manifest["train_sha256"])
print("Resolved DATA_DIR:", DATA_DIR)
train_path = DATA_DIR / "train.csv"
dl.verify_source(train_path, manifest)
train = pd.read_csv(train_path)
dl.validate_train(train)
for name, digest in manifest["membership_sha256"].items():
    if dl.file_hash(FOUNDATION_DIR / name) != digest:
        raise ValueError(f"Foundation membership file changed: {name}")
split = pd.read_csv(FOUNDATION_DIR / "holdout_split_seed9.csv")
train_idx, valid_idx = dl.holdout_indices(train, split)
print("Foundation:", manifest["foundation_id"])
print("Runtime:", dl.environment())


## 2.10.1 — Kiểm tra OOF và tính metric theo fold

OOF phải có đúng ID/nhãn/fold của foundation. Tính AUC trên từng fold và toàn bộ OOF riêng biệt; không nhầm hai đại lượng.
Chọn mô hình là quá trình dùng nhãn OOF, nên điểm tốt nhất sau lựa chọn có thể lạc quan; không gọi đây là điểm test độc lập.


In [ ]:
OUTPUT_DIR = Path("/kaggle/working/task_2_10_dl_ensemble")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MIN_MEAN_AUC_GAIN = 0.0001
MIN_IMPROVED_FOLDS = 3
cv_path = dl.find_input("cv_manifest.json")
cv = dl.json_read(cv_path)
if cv["foundation_id"] != manifest["foundation_id"] or cv["source_sha256"] != manifest["train_sha256"]:
    raise ValueError("CV output belongs to another foundation or dataset.")
fold_table = pd.read_csv(FOUNDATION_DIR / "cv_folds_seed9.csv")
fold_ids = fold_table.set_index(dl.ID).loc[train[dl.ID], "fold"].to_numpy(dtype=np.int64)
probabilities = {}
for key, model in cv["models"].items():
    path = cv_path.parent / model["oof_file"]
    if dl.file_hash(path) != model["oof_sha256"]:
        raise ValueError("OOF checksum mismatch.")
    frame = dl.prediction_frame(path, train)
    if not np.array_equal(frame["fold"].to_numpy(), fold_ids):
        raise ValueError("OOF folds differ from foundation.")
    probabilities[key] = frame["probability"].to_numpy(dtype=np.float64)

def evaluate(weights):
    if not np.isclose(sum(weights.values()), 1.0) or any(w < 0 for w in weights.values()):
        raise ValueError("Ensemble weights must be non-negative and sum to one.")
    probability = sum(weights[k] * probabilities[k] for k in weights)
    fold_auc, fold_ap = [], []
    for fold in range(5):
        mask = fold_ids == fold
        metric = dl.scores(train.loc[mask, dl.TARGET], probability[mask])
        fold_auc.append(metric["roc_auc"])
        fold_ap.append(metric["average_precision"])
    return {
        "weights": weights, "fold_auc": fold_auc,
        "mean_fold_auc": float(np.mean(fold_auc)),
        "std_fold_auc": float(np.std(fold_auc, ddof=1)),
        "mean_fold_ap": float(np.mean(fold_ap)),
        "pooled_oof": dl.scores(train[dl.TARGET], probability),
    }

def qualifies(candidate, reference):
    deltas = np.array(candidate["fold_auc"]) - np.array(reference["fold_auc"])
    return (float(deltas.mean()) >= MIN_MEAN_AUC_GAIN
            and int((deltas > 0).sum()) >= MIN_IMPROVED_FOLDS)


## 2.10.2 — Mô hình đơn trước, ensemble sau

Duyệt các mô hình đơn so với baseline; chọn model đủ điều kiện có mean AUC/AP cao nhất.
Blend phải cải thiện so với mô hình đơn đã chọn, không chỉ hơn một mô hình yếu. Không thử hàng trăm trọng số để dò OOF.


In [ ]:
single = {key: evaluate({key: 1.0}) for key in probabilities}
baseline = single[cv["baseline_id"]]
eligible = [baseline] + [r for k, r in single.items()
                         if k != cv["baseline_id"] and qualifies(r, baseline)]
selected = max(eligible, key=lambda r: (r["mean_fold_auc"], r["mean_fold_ap"]))
single_reference = selected
evaluations = list(single.values())

ordered_ids = sorted(single, key=lambda k: (
    single[k]["mean_fold_auc"], single[k]["mean_fold_ap"]), reverse=True)
if len(ordered_ids) >= 2:
    left, right = ordered_ids[:2]
    for alpha in [0.25, 0.50, 0.75]:
        blend = evaluate({left: alpha, right: 1.0 - alpha})
        evaluations.append(blend)
        if qualifies(blend, single_reference) and (
            blend["mean_fold_auc"], blend["mean_fold_ap"]
        ) > (selected["mean_fold_auc"], selected["mean_fold_ap"]):
            selected = blend

selected_probability = sum(
    weight * probabilities[key] for key, weight in selected["weights"].items())
pd.DataFrame({dl.ID: train[dl.ID], dl.TARGET: train[dl.TARGET], "fold": fold_ids,
              "probability": selected_probability}).to_csv(OUTPUT_DIR / "selected_oof.csv", index=False)
table = pd.DataFrame([{
    "weights": json.dumps(r["weights"], sort_keys=True),
    "mean_fold_auc": r["mean_fold_auc"], "std_fold_auc": r["std_fold_auc"],
    "mean_fold_ap": r["mean_fold_ap"], "pooled_oof_auc": r["pooled_oof"]["roc_auc"],
    "mean_delta_vs_baseline": r["mean_fold_auc"] - baseline["mean_fold_auc"],
    "selected": r["weights"] == selected["weights"],
} for r in evaluations]).sort_values("mean_fold_auc", ascending=False)
table.to_csv(OUTPUT_DIR / "ensemble_comparison.csv", index=False)
selection = {
    "foundation_id": manifest["foundation_id"], "source_sha256": manifest["train_sha256"],
    "cv_manifest_sha256": dl.file_hash(cv_path), "engine_sha256": manifest["engine_sha256"],
    "weights": selected["weights"], "selected_metrics": selected,
    "baseline_metrics": baseline,
    "policy": {"min_mean_auc_gain": MIN_MEAN_AUC_GAIN, "min_improved_folds": MIN_IMPROVED_FOLDS,
               "rule": "Single vs baseline; ensemble vs selected single. Practical, not statistical."},
    "models": {key: cv["models"][key] for key in selected["weights"]},
    "inference": "Mean of five fold checkpoints per config, then blend using frozen weights.",
    "test_used_for_selection": False, "leaderboard_used_for_selection": False,
    "limitations": cv["limitations"] + " OOF weights also selected on these folds.",
}
dl.json_write(OUTPUT_DIR / "ensemble_selection.json", selection)
display(table)
print("Frozen weights:", selection["weights"])
